# Customizing Models and Training Algorithms

In [3]:
def huber_fn(y_true, y_pred):
  error = y_true - y_pred
  is_small_error = tf.abs(error) < 1
  squared_loss = tf.square(error) / 2
  linear_loss = tf.abs(error) - 0.5
  return tf.where(is_small_error, squared_loss, linear_loss)


  """
Huber loss is a loss used in regression because it protects the model from outliers.

It combines:

MSE (squared loss) when the error is small

MAE (absolute loss) when the error is large
"""
#model.compile(loss=huber_fn, optimizer="nadam")
#model.fit(X_train, y_train, [...])


##### when load model with custom loss fun use:
# model = tf.keras.models.load_model("my_model_with_a_custom_loss", custom_objects={"huber_fn": huber_fn})

In [ ]:
# custom_layers_ch12.py
# Full chapter 12 implementation: custom activations, initializers,
# regularizers, constraints, metrics, layers, models, and examples.

import tensorflow as tf

# -------------------------------
# Custom Activation
# -------------------------------
def my_softplus(z):
    return tf.math.log(1.0 + tf.exp(z))

# -------------------------------
# Custom Initializer
# -------------------------------
def my_glorot_initializer(shape, dtype=tf.float32):
    stddev = tf.sqrt(2.0 / (shape[0] + shape[1]))
    return tf.random.normal(shape, stddev=stddev, dtype=dtype)

# -------------------------------
# Custom Regularizer
# -------------------------------
def my_l1_regularizer(weights):
    return tf.reduce_sum(tf.abs(0.01 * weights))

# -------------------------------
# Custom Constraint
# -------------------------------
def my_positive_weights(weights):
    return tf.where(weights < 0.0, tf.zeros_like(weights), weights)

# -------------------------------
# Custom Regularizer Class
# -------------------------------
class MyL1Regularizer(tf.keras.regularizers.Regularizer):
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, weights):
        return tf.reduce_sum(tf.abs(weights)) * self.factor

    def get_config(self):
        return {"factor": self.factor}




In [ ]:
# -------------------------------
# Custom Metric Class (Huber)
# -------------------------------
def create_huber(threshold=1.0):
    def huber_fn(y_true, y_pred):
        error = y_true - y_pred
        is_small = tf.abs(error) < threshold
        small_loss = tf.square(error) / 2
        big_loss = threshold * (tf.abs(error) - 0.5 * threshold)
        return tf.where(is_small, small_loss, big_loss)
    return huber_fn



In [ ]:
class HuberMetric(tf.keras.metrics.Metric):
    def __init__(self, threshold=1.0, **kwargs):
        super().__init__(**kwargs)
        self.threshold = threshold
        self.huber_fn = create_huber(threshold)
        self.total = self.add_weight("total", initializer="zeros")
        self.count = self.add_weight("count", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        values = self.huber_fn(y_true, y_pred)
        self.total.assign_add(tf.reduce_sum(values))
        self.count.assign_add(tf.cast(tf.size(y_true), tf.float32))

    def result(self):
        return self.total / self.count

    def get_config(self):
        base = super().get_config()
        return {**base, "threshold": self.threshold}

# -------------------------------
# Custom Dense Layer
# -------------------------------
class MyDense(tf.keras.layers.Layer):
    def __init__(self, units, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = tf.keras.activations.get(activation)

    def build(self, input_shape):
        self.kernel = self.add_weight(
            name="kernel",
            shape=(input_shape[-1], self.units),
            initializer="glorot_uniform"
        )
        self.bias = self.add_weight(
            name="bias",
            shape=(self.units,),
            initializer="zeros"
        )
        super().build(input_shape)

    def call(self, x):
        z = tf.matmul(x, self.kernel) + self.bias
        return self.activation(z)

# -------------------------------
# Residual Block
# -------------------------------
class ResidualBlock(tf.keras.layers.Layer):
    def __init__(self, n_layers, n_neurons, **kwargs):
        super().__init__(**kwargs)
        self.hidden_layers = [
            tf.keras.layers.Dense(n_neurons, activation="relu")
            for _ in range(n_layers)
        ]

    def call(self, inputs):
        z = inputs
        for layer in self.hidden_layers:
            z = layer(z)
        return inputs + z

# -------------------------------
# Custom Model with Residual Blocks
# -------------------------------
class ResidualRegressor(tf.keras.Model):
    def __init__(self, output_dim):
        super().__init__()
        self.hidden1 = tf.keras.layers.Dense(30, activation="relu")
        self.block1 = ResidualBlock(2, 30)
        self.block2 = ResidualBlock(2, 30)
        self.out_layer = tf.keras.layers.Dense(output_dim)

    def call(self, inputs):
        z = self.hidden1(inputs)
        for _ in range(3):
            z = self.block1(z)
        z = self.block2(z)
        return self.out_layer(z)

# -------------------------------
# Custom Model Using Loss Based on Internals
# -------------------------------
class ReconstructingRegressor(tf.keras.Model):
    def __init__(self, output_dim):
        super().__init__()
        self.hidden_layers = [
            tf.keras.layers.Dense(30, activation="relu")
            for _ in range(5)
        ]
        self.output_layer = tf.keras.layers.Dense(output_dim)
        self.reconstruction_mean = tf.keras.metrics.Mean(name="reconstruction_error")

    def build(self, input_shape):
        n_inputs = input_shape[-1]
        self.reconstruct = tf.keras.layers.Dense(n_inputs)

    def call(self, inputs, training=False):
        z = inputs
        for layer in self.hidden_layers:
            z = layer(z)

        reconstruction = self.reconstruct(z)
        recon_loss = tf.reduce_mean(tf.square(reconstruction - inputs))
        self.add_loss(0.05 * recon_loss)

        if training:
            self.add_metric(self.reconstruction_mean(recon_loss))

        return self.output_layer(z)

# -------------------------------
# End of File
# -------------------------------